# Multi-Bot  — Convoy System

**Replaces the sporadic independent-search model with a locked convoy chain.**

## What changed 

### 1. Convoy chain (req 1)
- Any bot that holds the cube for ≥ `convoy_announce_min_frames` consecutive frames **announces** itself as convoy leader.
- The brain builds a chain: `[leader, follower-1, follower-2, …]` ordered by each bot's last-known cube distance.
- Each follower exclusively follows the bot **one slot ahead** at exactly `convoy_spacing_meters` (0.20 m).
- Followers **stop the moment** their follow-target stops — no independent roaming ever.

### 2. Target locked; reactive tracking (req 2 + 2a)
- Leader uses **raw** (unsmoothed) `cube_distance_meters` for motor commands — reacts the same frame the target moves.
- `brain_distance_ema_alpha` 0.60 → **0.30** and `cube_lock_grace_frames` 4 → **2** for faster reacquisition.
- When target moves farther: `dist_error` grows → leader speeds up → followers chase the widening gap.

### 3. Dynamic leader handoff (req 3)
- If the leader is blind for > `convoy_leader_handoff_frames` frames (0.8 s), `brain_convoy_try_handoff()` promotes whichever other bot has the freshest cube sighting.
- Old leader rejoins chain as a follower. New leader immediately approaches.


## §1. Imports and bot registry

**Edit IPs and uncomment the bots you're using.** For the first multi-bot test, leave just 2 bots active and comment out the rest. Each bot must be running the server with the matching `my_own_color`.


In [ ]:
import random
import requests
import time
import threading
import math
import io
from urllib.parse import urljoin
from IPython.display import display
import ipywidgets as widgets
from PIL import Image as PILImage, ImageDraw
import base64

# ── EDIT THIS ────────────────────────────────────────────────────────
# Comment out bots you're not using right now. Start with just 2 for first test.
bot_registry = [
    {'bot_id': 'bot_1', 'ip_address': '194.47.156.39',  'assigned_color': 'blue'},
    {'bot_id': 'bot_2', 'ip_address': '194.47.156.43', 'assigned_color': 'purple'},
    #{'bot_id': 'bot_3', 'ip_address': '194.47.156.201',  'assigned_color': 'orange'},
    #{'bot_id': 'bot_4', 'ip_address': '194.47.156.213',  'assigned_color': 'yellow'},
]
bot_http_port = 8080
# ──────────────────────────────────────────────────────────────────────

def base_url_for_bot(record):
    return f"http://{record['ip_address']}:{bot_http_port}"

http_session_by_bot_id = {b['bot_id']: requests.Session() for b in bot_registry}
bot_record_by_id = {b['bot_id']: b for b in bot_registry}
known_bot_ids = [b['bot_id'] for b in bot_registry]
color_to_bot_id = {b['assigned_color']: b['bot_id'] for b in bot_registry}

display_color_hex_for_color = {
    'red':    '#ff0000', 'orange': '#ff8c00', 'yellow': '#ffd000',
    'blue':   '#0064ff', 'purple': '#c800c8',
}

print(f"Registered {len(bot_registry)} bot(s):")
for b in bot_registry:
    print(f"  {b['bot_id']:<8s}  {b['ip_address']:<18s}  color={b['assigned_color']}")


## §2. Parameters

All v5 parameters carry over unchanged. New parameters for peer avoidance are at the bottom.


In [ ]:
control_parameters_default = {
    # ── Approach ─────────────────────────────────────────────────
    'target_stopping_distance_meters':            0.30,
    'approach_complete_zone_meters':              0.02,
    'approach_bearing_tolerance':                 0.18,
    'approach_bearing_tolerance_final':           0.12,
    'forward_creep_when_bearing_off':             0.20,

    # ── Frame-fill arrival ───────────────────────────────────────
    'arrived_cube_fills_frame_ratio':             0.15,
    'arrived_cube_fills_frame_hysteresis':        0.05,
    'close_range_fill_ratio':                     0.10,
    'close_range_bearing_tolerance':              0.35,

    # ── Un-arrival ───────────────────────────────────────────────
    'unarrive_distance_threshold_meters':         0.30,
    'unarrive_consecutive_frames':                3,      # Stage 6: was 20; resume fast when target moves/leaves

    # ── Motor budget ─────────────────────────────────────────────
    'maximum_motor_speed':                        0.18,
    'motor_stiction_floor':                       0.10,
    'forward_minimum_to_move':                    0.10,
    'turn_deadband_in_local_x':                   0.04,

    # ── Cube tracking gains ──────────────────────────────────────
    'forward_pull_gain_per_meter':                1.5,
    'turn_pull_gain_per_x_normalized':            0.5,

    # ── Arc-not-pivot ────────────────────────────────────────────
    'target_arc_minimum_wheel_ratio':             0.20,
    'turn_to_wheel_diff_scale':                   0.5,

    # ── Search / lock ────────────────────────────────────────────
    'lost_rotation_speed':                        0.10,
    'cube_lock_grace_frames':                     2,      # Stage 6: was 4; faster reaction (req 2a)

    # ── Loop / network ───────────────────────────────────────────
    'control_loop_period_seconds':                0.10,
    'http_request_timeout_seconds':               0.6,
    'command_watchdog_ttl_ms':                    500,

    # ── Peer repulsion ───────────────────────────────────────────
    'peer_repulsion_range_meters':                1.00,
    'peer_repulsion_weight':                      0.6,
    'peer_repulsion_falloff_exponent':            2.0,
    'peer_critical_distance_meters':              0.15,   # Stage 6: tightened (convoy gap is 0.20 m)
    'peer_minimum_useful_distance_meters':        0.05,

    # ── Caution slowdown ─────────────────────────────────────────
    'peer_slowdown_range_meters':                 0.80,
    'peer_slowdown_x_normalized_band':            0.40,
    'peer_slowdown_minimum_factor':               0.30,

    # ── Tracker dead-band (fill-scaled) ──────────────────────────
    'tracker_turn_gain':                          0.7,
    'tracker_turn_speed_max':                     0.12,
    'tracker_dead_band_far':                      0.06,
    'tracker_dead_band_mid':                      0.15,
    'tracker_dead_band_close':                    0.30,
    'tracker_fill_threshold_mid':                 0.15,
    'tracker_fill_threshold_close':               0.30,

    # ── Camera ────────────────────────────────────────────────────
    'camera_horizontal_fov_degrees':              62.0,

    # ── Logging ──────────────────────────────────────────────────
    'log_every_n_iterations':                     5,

    # ── Brain smoothing + leader hysteresis ──────────────────────
    'brain_distance_ema_alpha':                   0.30,   # Stage 6: was 0.60; more reactive (req 2a)
    'brain_distance_stale_seconds':               1.5,
    'leader_change_minimum_distance_advantage':   0.25,
    'leader_change_minimum_relative_advantage':   0.20,

    # ── Stall detection (Stage 5.1) ──────────────────────────────
    'stall_detection_pixel_threshold':            5.0,
    'stall_detection_frames_threshold':           15,
    'stall_recovery_reverse_speed':               0.15,
    'stall_recovery_reverse_frames':              3,

    # ── PSO search (used only before convoy is announced) ────────
    'pso_inertia_w':          0.12,
    'pso_cognitive_c1':       1.2,
    'pso_social_c2':          1.2,
    'pso_max_velocity_x':     0.15,
    'pso_max_velocity_y':     0.10,
    'search_forward_speed':   0.11,

    # ── Stage 6: Convoy system ────────────────────────────────────
    # Gap to maintain between consecutive convoy members (req 1).
    'convoy_spacing_meters':          0.20,
    # Max speed a follower uses to close a gap.
    'convoy_follow_speed_max':        0.20,
    # speed = gain * (actual_gap - spacing); clamped to [stiction, max].
    'convoy_follow_gain':             1.8,
    # Proportional turn gain when following a peer (0-1 normalised bearing).
    'convoy_turn_gain':               0.65,
    # Frames leader must continuously see the cube before announcing convoy.
    'convoy_announce_min_frames':     3,
    # Frames the leader may be blind before handing off leadership (req 3).
    'convoy_leader_handoff_frames':   8,
    # Frames a follower waits (bot-ahead unseen) before it starts searching.
    'convoy_follower_search_grace_frames': 3,
}


control_parameter_overrides_per_bot = {}

def param(bot_id, key):
    ov = control_parameter_overrides_per_bot.get(bot_id, {})
    return ov[key] if key in ov else control_parameters_default[key]

def set_override(bot_id, key, value):
    if bot_id not in control_parameter_overrides_per_bot:
        control_parameter_overrides_per_bot[bot_id] = {}
    control_parameter_overrides_per_bot[bot_id][key] = value

print('Parameters loaded (Stage 6 — convoy system, reactive tracking).')

## §3. Network helpers

In [ ]:
# Per-bot frame dimensions, populated from /health
frame_width_by_bot_id  = {bid: 3000 for bid in known_bot_ids}
frame_height_by_bot_id = {bid: 1100 for bid in known_bot_ids}


def fetch_state(bot_id):
    '''Stage 5.1: also return frame_difference_value for stall detection.'''
    record = bot_record_by_id[bot_id]
    session = http_session_by_bot_id[bot_id]
    timeout = param(bot_id, 'http_request_timeout_seconds')
    t0 = time.time()
    try:
        r = session.get(urljoin(base_url_for_bot(record), '/state'), timeout=timeout)
        latency = time.time() - t0
        if r.status_code != 200:
            return False, None, [], latency, 0.0
        payload = r.json()
    except requests.RequestException:
        return False, None, [], time.time() - t0, 0.0
    peers = list(payload.get('peers', []))
    frame_diff = float(payload.get('frame_difference_value', 0.0))
    if not payload.get('cube_visible', False):
        return True, None, peers, latency, frame_diff
    cube = {
        'cube_x_normalized':    float(payload['cube_x_normalized']),
        'cube_y_normalized':    float(payload['cube_y_normalized']),
        'cube_distance_meters': float(payload['cube_distance_meters']),
    }
    bbox = payload.get('cube_bounding_box')
    if bbox is not None and len(bbox) >= 4:
        cube['cube_bounding_box'] = tuple(int(v) for v in bbox[:4])
    return True, cube, peers, latency, frame_diff


def send_command(bot_id, left, right, ttl_ms=None):
    record = bot_record_by_id[bot_id]
    session = http_session_by_bot_id[bot_id]
    timeout = param(bot_id, 'http_request_timeout_seconds')
    if ttl_ms is None:
        ttl_ms = param(bot_id, 'command_watchdog_ttl_ms')
    try:
        r = session.post(
            urljoin(base_url_for_bot(record), '/command'),
            json={'left_motor_speed': float(left),
                  'right_motor_speed': float(right),
                  'command_ttl_milliseconds': int(ttl_ms)},
            timeout=timeout)
        return r.status_code == 200
    except requests.RequestException:
        return False


def stop_bot(bot_id):
    return send_command(bot_id, 0.0, 0.0, ttl_ms=100)


def stop_all_bots():
    for bid in known_bot_ids:
        stop_bot(bid)


def fetch_health(bot_id):
    record = bot_record_by_id[bot_id]
    session = http_session_by_bot_id[bot_id]
    try:
        r = session.get(urljoin(base_url_for_bot(record), '/health'), timeout=1.5)
        return r.json() if r.status_code == 200 else None
    except requests.RequestException:
        return None

print('Network helpers ready.')


## §4. Connection self-test

In [ ]:
def run_self_test():
    print(f'Self-testing {len(bot_registry)} bot(s)...')
    print()
    any_bad = False
    for record in bot_registry:
        bid = record['bot_id']
        h = fetch_health(bid)
        if h is None:
            print(f"  {bid:<8s} ❌ unreachable")
            any_bad = True
            continue
        rate = h.get('detection_loop_hz', 0)
        col  = h.get('bot_own_color')
        fw   = h.get('cropped_frame_width')
        fh   = h.get('cropped_frame_height')
        if fw is not None:
            frame_width_by_bot_id[bid]  = int(fw)
        if fh is not None:
            frame_height_by_bot_id[bid] = int(fh)
        color_match = '✓' if col == record['assigned_color'] else f"⚠ bot says '{col}'"
        rate_flag = '✓' if rate >= 5.0 else ('⚠ marginal' if rate >= 3.0 else '❌ too slow')
        print(f"  {bid:<8s}  rate={rate:4.1f}Hz {rate_flag}  color={col} {color_match}  frame={fw}×{fh}")
        if rate < 3.0 or col != record['assigned_color']:
            any_bad = True
    print()
    print('✓ All good.' if not any_bad else '⚠ Fix issues above before continuing.')

run_self_test()


## §5. Controller — Stage 6 Convoy

### Architecture

```
PHASE 1 — SEARCH  (before convoy announced)
  All bots search independently with PSO arcs.
  First bot to hold cube ≥ convoy_announce_min_frames becomes leader,
  announces convoy, chain is built ordered by cube distance.

PHASE 2 — CONVOY  (after convoy announced)
  LEADER   → approaches target reactively using raw distance (req 2).
             Grace frames → PSO search → handoff if blind too long (req 3).
  FOLLOWER → follows the bot one slot ahead at convoy_spacing_meters (0.20 m).
             Stops immediately when follow-target stops (req 1).
             Does NOT search independently — ever.
```

### Key brain convoy functions
- `brain_convoy_tick(bot_id, has_cube)` — counts sightings pre-announcement; announces when threshold reached
- `brain_convoy_try_handoff(lost_leader_id, missed_frames)` — promotes freshest cube-seer to leader
- `brain_get_convoy_info()` — thread-safe snapshot of announced/leader/chain


In [ ]:
# ══════════════════════════════════════════════════════════════════
# BRAIN SHARED STATE
# ══════════════════════════════════════════════════════════════════
brain_lock  = threading.Lock()
brain_state = {
    # Per-bot raw + smoothed cube distance (used for chain ordering).
    'cube_distance_by_bot_id':            {bid: None  for bid in known_bot_ids},
    'cube_distance_smoothed_by_bot_id':   {bid: None  for bid in known_bot_ids},
    'cube_distance_timestamp_by_bot_id':  {bid: 0.0   for bid in known_bot_ids},
    'arrived_bot_ids':                    set(),

    # ── Stage 6: Convoy ───────────────────────────────────────────
    'convoy_announced':        False,   # True once a leader has locked on
    'convoy_chain':            [],      # [leader_id, follower1_id, follower2_id, …]
    'convoy_leader_id':        None,    # The designated convoy leader
    'convoy_candidate_frames': {bid: 0 for bid in known_bot_ids},  # pre-announcement counter

    # PSO state (pre-announcement search only).
    'pso_vel_x_by_bot_id':    {bid: 0.0    for bid in known_bot_ids},
    'pso_vel_y_by_bot_id':    {bid: 0.0    for bid in known_bot_ids},
    'pso_best_x_by_bot_id':   {bid: 0.0    for bid in known_bot_ids},
    'pso_best_dist_by_bot_id':{bid: 9999.0 for bid in known_bot_ids},
    'pso_global_best_x':      0.0,
    'pso_global_best_dist':   9999.0,
}


# ── Cube distance publish (EMA for chain ordering only) ───────────
def brain_publish_cube_distance(bot_id, distance_or_none):
    alpha = param(bot_id, 'brain_distance_ema_alpha')
    now   = time.time()
    with brain_lock:
        brain_state['cube_distance_by_bot_id'][bot_id]           = distance_or_none
        brain_state['cube_distance_timestamp_by_bot_id'][bot_id] = now
        if distance_or_none is None:
            return
        prior = brain_state['cube_distance_smoothed_by_bot_id'].get(bot_id)
        smoothed = (float(distance_or_none) if prior is None
                    else alpha * prior + (1.0 - alpha) * float(distance_or_none))
        brain_state['cube_distance_smoothed_by_bot_id'][bot_id] = smoothed

def brain_mark_arrived(bot_id):
    with brain_lock:
        brain_state['arrived_bot_ids'].add(bot_id)

def brain_mark_unarrived(bot_id):
    with brain_lock:
        brain_state['arrived_bot_ids'].discard(bot_id)


# ── Stage 6: Convoy announcement ─────────────────────────────────
def brain_convoy_tick(bot_id, has_cube):
    """Call every frame from every bot.
    Returns (is_convoy_leader, convoy_announced).

    Pre-announcement: counts consecutive sightings per bot.
      When any bot reaches convoy_announce_min_frames it becomes leader,
      builds the chain ordered by smoothed cube distance, and announces.
    Post-announcement: returns current role without modification.
    """
    min_frames = control_parameters_default['convoy_announce_min_frames']
    with brain_lock:
        announced = brain_state['convoy_announced']
        leader_id = brain_state['convoy_leader_id']

        if not announced:
            # Track candidate frames for this bot.
            if has_cube:
                brain_state['convoy_candidate_frames'][bot_id] += 1
            else:
                brain_state['convoy_candidate_frames'][bot_id] = 0

            if brain_state['convoy_candidate_frames'][bot_id] >= min_frames:
                # This bot becomes leader. Build chain.
                smoothed_map = brain_state['cube_distance_smoothed_by_bot_id']
                others = [b for b in known_bot_ids if b != bot_id]
                others.sort(key=lambda b: (smoothed_map.get(b) or 9999.0))
                chain = [bot_id] + others
                brain_state['convoy_announced'] = True
                brain_state['convoy_chain']     = chain
                brain_state['convoy_leader_id'] = bot_id
                print(f"  [brain] CONVOY ANNOUNCED — leader={bot_id}, chain={chain}")
                return True, True

            return False, False

        # Already announced — just report role.
        return (leader_id == bot_id), True


def brain_convoy_try_handoff(lost_leader_id, missed_frames):
    """Called each frame the convoy leader can't see the cube.
    Returns new_leader_id once missed_frames >= threshold, else None.
    """
    threshold   = control_parameters_default['convoy_leader_handoff_frames']
    stale_limit = control_parameters_default['brain_distance_stale_seconds']
    if missed_frames < threshold:
        return None
    now = time.time()
    with brain_lock:
        if brain_state['convoy_leader_id'] != lost_leader_id:
            return None  # Already handed off by another thread.
        smoothed_map = brain_state['cube_distance_smoothed_by_bot_id']
        ts_map       = brain_state['cube_distance_timestamp_by_bot_id']
        candidates   = []
        for bid in known_bot_ids:
            if bid == lost_leader_id:
                continue
            d  = smoothed_map.get(bid)
            ts = ts_map.get(bid, 0.0)
            if d is not None and (now - ts) < stale_limit:
                candidates.append((d, bid))
        if not candidates:
            return None  # No one sees it; leader keeps searching.
        candidates.sort()
        new_leader_id = candidates[0][1]
        old_chain = brain_state['convoy_chain']
        remaining = [b for b in old_chain if b != new_leader_id]
        new_chain = [new_leader_id] + remaining
        brain_state['convoy_chain']     = new_chain
        brain_state['convoy_leader_id'] = new_leader_id
        print(f"  [brain] CONVOY HANDOFF: {lost_leader_id} → {new_leader_id}, "
              f"chain={new_chain}")
        return new_leader_id


def brain_get_convoy_info():
    """Thread-safe snapshot: (announced, leader_id, chain)."""
    with brain_lock:
        return (brain_state['convoy_announced'],
                brain_state['convoy_leader_id'],
                list(brain_state['convoy_chain']))


def brain_get_status_snapshot():
    with brain_lock:
        return {
            'leader':    brain_state['convoy_leader_id'],
            'arrived':   set(brain_state['arrived_bot_ids']),
            'distances': dict(brain_state['cube_distance_smoothed_by_bot_id']),
            'chain':     list(brain_state['convoy_chain']),
            'announced': brain_state['convoy_announced'],
        }


# ── PSO (pre-announcement only) ───────────────────────────────────
def brain_publish_pso_sighting(bot_id, x_normalized, distance_meters):
    with brain_lock:
        if distance_meters < brain_state['pso_best_dist_by_bot_id'][bot_id]:
            brain_state['pso_best_x_by_bot_id'][bot_id]    = float(x_normalized)
            brain_state['pso_best_dist_by_bot_id'][bot_id] = float(distance_meters)
        best_bot = min(known_bot_ids,
                       key=lambda b: brain_state['pso_best_dist_by_bot_id'][b])
        brain_state['pso_global_best_x']    = brain_state['pso_best_x_by_bot_id'][best_bot]
        brain_state['pso_global_best_dist'] = brain_state['pso_best_dist_by_bot_id'][best_bot]


def brain_compute_pso_velocity(bot_id, current_x, current_dist):
    W   = control_parameters_default['pso_inertia_w']
    C1  = control_parameters_default['pso_cognitive_c1']
    C2  = control_parameters_default['pso_social_c2']
    mvx = control_parameters_default['pso_max_velocity_x']
    mvy = control_parameters_default['pso_max_velocity_y']
    with brain_lock:
        vx = brain_state['pso_vel_x_by_bot_id'][bot_id]
        vy = brain_state['pso_vel_y_by_bot_id'][bot_id]
        px = brain_state['pso_best_x_by_bot_id'][bot_id]
        pd = brain_state['pso_best_dist_by_bot_id'][bot_id]
        gx = brain_state['pso_global_best_x']
        gd = brain_state['pso_global_best_dist']
        r1x, r1y = random.random(), random.random()
        r2x, r2y = random.random(), random.random()
        new_vx = W*vx + C1*r1x*(px - current_x)   + C2*r2x*(gx - current_x)
        new_vy = W*vy + C1*r1y*(pd - current_dist) + C2*r2y*(gd - current_dist)
        new_vx = max(-mvx, min(mvx, new_vx))
        new_vy = max(-mvy, min(mvy, new_vy))
        brain_state['pso_vel_x_by_bot_id'][bot_id] = new_vx
        brain_state['pso_vel_y_by_bot_id'][bot_id] = new_vy
        return new_vx, new_vy


def brain_reset():
    with brain_lock:
        for bid in known_bot_ids:
            brain_state['cube_distance_by_bot_id'][bid]           = None
            brain_state['cube_distance_smoothed_by_bot_id'][bid]  = None
            brain_state['cube_distance_timestamp_by_bot_id'][bid] = 0.0
            brain_state['convoy_candidate_frames'][bid] = 0
            brain_state['pso_vel_x_by_bot_id'][bid]     = 0.0
            brain_state['pso_vel_y_by_bot_id'][bid]     = 0.0
            brain_state['pso_best_x_by_bot_id'][bid]    = 0.0
            brain_state['pso_best_dist_by_bot_id'][bid] = 9999.0
        brain_state['arrived_bot_ids']   = set()
        brain_state['convoy_announced']  = False
        brain_state['convoy_chain']      = []
        brain_state['convoy_leader_id']  = None
        brain_state['pso_global_best_x']    = 0.0
        brain_state['pso_global_best_dist'] = 9999.0
    print('Brain reset (Stage 6: convoy + PSO cleared).')


# ══════════════════════════════════════════════════════════════════
# PER-BOT STATE
# ══════════════════════════════════════════════════════════════════
def _default_bot_state():
    return {
        'last_seen_cube_x_normalized': 0.0,
        'consecutive_missed_frames':   0,
        'previous_cube_distance':      None,
        'last_motor_command_left':     0.0,
        'last_motor_command_right':    0.0,
        'arrived_by_fill_latched':     False,
        'frames_cube_departed_count':  0,
        'frames_cube_missing_count':   0,
        'stall_low_motion_frames':     0,
        'stall_recovery_frames_left':  0,
        'follower_lost_target_frames': 0,
    }

bot_state = {bid: _default_bot_state() for bid in known_bot_ids}

def reset_bot_states_for_new_run():
    for bid in known_bot_ids:
        bot_state[bid] = _default_bot_state()
    print('Per-bot controller state reset (Stage 6).')


# ══════════════════════════════════════════════════════════════════
# GEOMETRY / MOTOR HELPERS  (unchanged)
# ══════════════════════════════════════════════════════════════════
def compute_adaptive_arc_ratio(bot_id, forward_target_motor_units, turn_intensity):
    target_ratio = (1.0 - (1.0 - param(bot_id, 'target_arc_minimum_wheel_ratio'))
                    * min(1.0, abs(turn_intensity)))
    if abs(forward_target_motor_units) > 1e-6:
        stiction = param(bot_id, 'motor_stiction_floor')
        minimum_safe_ratio = stiction / abs(forward_target_motor_units)
        target_ratio = max(target_ratio, minimum_safe_ratio)
    return min(target_ratio, 1.0)


def arc_wheel_mapping(bot_id, forward_target, turn_intensity):
    max_speed = param(bot_id, 'maximum_motor_speed')
    stiction  = param(bot_id, 'motor_stiction_floor')
    f, t = forward_target, turn_intensity
    if abs(f) >= stiction:
        slower_ratio = compute_adaptive_arc_ratio(bot_id, f, t)
        if   t > 0: left, right = f, f * slower_ratio
        elif t < 0: left, right = f * slower_ratio, f
        else:       left = right = f
    elif abs(t) >= 0.05:
        rot = max(stiction, abs(t) * max_speed)
        left, right = (+rot, -rot) if t > 0 else (-rot, +rot)
        slower_ratio = 0.0
    else:
        left = right = 0.0
        slower_ratio = 1.0
    left  = max(-max_speed, min(max_speed, left))
    right = max(-max_speed, min(max_speed, right))
    return left, right, slower_ratio


def compute_frame_fill_ratio(bot_id, cube_detection):
    bbox = cube_detection.get('cube_bounding_box') if cube_detection else None
    if bbox is None or len(bbox) < 4:
        return None
    return bbox[2] / max(1, frame_width_by_bot_id[bot_id])


def bearing_to_local_vector(x_normalized, magnitude, fov_degrees):
    half_fov    = math.radians(fov_degrees) / 2.0
    bearing_rad = x_normalized * half_fov
    return (magnitude * math.sin(bearing_rad), magnitude * math.cos(bearing_rad))


def compute_peer_repulsion_vector(bot_id, peer_x_normalized, peer_distance_meters):
    range_m    = param(bot_id, 'peer_repulsion_range_meters')
    min_useful = param(bot_id, 'peer_minimum_useful_distance_meters')
    if peer_distance_meters >= range_m or peer_distance_meters < min_useful:
        return (0.0, 0.0)
    falloff   = param(bot_id, 'peer_repulsion_falloff_exponent')
    weight    = param(bot_id, 'peer_repulsion_weight')
    fov       = param(bot_id, 'camera_horizontal_fov_degrees')
    closeness = max(0.0, (range_m - peer_distance_meters) / range_m)
    magnitude = (closeness ** falloff) * weight
    px, py    = bearing_to_local_vector(peer_x_normalized, 1.0, fov)
    return (-magnitude * px, -magnitude * py)


def aggregate_peer_repulsion(bot_id, peers_list, exclude_color=None):
    own_color  = bot_record_by_id[bot_id]['assigned_color']
    crit_dist  = param(bot_id, 'peer_critical_distance_meters')
    min_useful = param(bot_id, 'peer_minimum_useful_distance_meters')
    sum_vx = sum_vy = 0.0
    critical_peer   = None
    visible_summary = []
    relevant_peers  = []
    for p in peers_list:
        color = p.get('peer_color')
        if color is None or color == own_color:
            continue
        if exclude_color is not None and color == exclude_color:
            continue
        d = float(p.get('peer_distance_meters', 999.0))
        x = float(p.get('peer_x_normalized', 0.0))
        if d < min_useful or d > 5.0:
            continue
        vx, vy = compute_peer_repulsion_vector(bot_id, x, d)
        sum_vx += vx
        sum_vy += vy
        visible_summary.append(f"{color}@{d:.1f}m")
        relevant_peers.append((color, x, d))
        if d < crit_dist:
            if critical_peer is None or d < critical_peer['d']:
                critical_peer = {'color': color, 'd': d, 'x': x}
    return sum_vx, sum_vy, critical_peer, visible_summary, relevant_peers


def compute_caution_slowdown_factor(bot_id, relevant_peers):
    slowdown_range = param(bot_id, 'peer_slowdown_range_meters')
    bearing_band   = param(bot_id, 'peer_slowdown_x_normalized_band')
    min_factor     = param(bot_id, 'peer_slowdown_minimum_factor')
    factor = 1.0
    for (color, x, d) in relevant_peers:
        if d >= slowdown_range or abs(x) >= bearing_band:
            continue
        proximity   = (slowdown_range - d) / slowdown_range
        this_factor = 1.0 - proximity * (1.0 - min_factor)
        factor *= this_factor
    return max(min_factor, min(1.0, factor)), []


def compute_tracker_dead_band(bot_id, fill_ratio):
    if fill_ratio is None:
        return param(bot_id, 'tracker_dead_band_far')
    if fill_ratio >= param(bot_id, 'tracker_fill_threshold_close'):
        return param(bot_id, 'tracker_dead_band_close')
    elif fill_ratio >= param(bot_id, 'tracker_fill_threshold_mid'):
        return param(bot_id, 'tracker_dead_band_mid')
    return param(bot_id, 'tracker_dead_band_far')


def compute_tracker_rotation(bot_id, cube_x_normalized, fill_ratio):
    dead_band = compute_tracker_dead_band(bot_id, fill_ratio)
    turn_gain = param(bot_id, 'tracker_turn_gain')
    turn_max  = param(bot_id, 'tracker_turn_speed_max')
    stiction  = param(bot_id, 'motor_stiction_floor')
    if abs(cube_x_normalized) < dead_band:
        return 0.0, 0.0
    abs_turn = max(stiction, min(turn_max, abs(turn_gain * cube_x_normalized) * turn_max))
    return (+abs_turn, -abs_turn) if cube_x_normalized > 0 else (-abs_turn, +abs_turn)


def update_unarrival_counters_and_maybe_unarrive(bot_id, cube_detection):
    st = bot_state[bot_id]
    if not st['arrived_by_fill_latched']:
        st['frames_cube_departed_count'] = 0
        st['frames_cube_missing_count']  = 0
        return False
    threshold_d = param(bot_id, 'unarrive_distance_threshold_meters')
    threshold_n = param(bot_id, 'unarrive_consecutive_frames')
    if cube_detection is None:
        st['frames_cube_missing_count'] += 1
    else:
        st['frames_cube_missing_count'] = 0
        if cube_detection['cube_distance_meters'] > threshold_d:
            st['frames_cube_departed_count'] += 1
        else:
            st['frames_cube_departed_count'] = 0
    for count_key, label in [('frames_cube_departed_count', 'departed'),
                              ('frames_cube_missing_count',  'missing')]:
        if st[count_key] >= threshold_n:
            print(f"  [{bot_id}] UNARRIVED (cube {label} for {st[count_key]} frames)")
            st['arrived_by_fill_latched']    = False
            st['frames_cube_departed_count'] = 0
            st['frames_cube_missing_count']  = 0
            brain_mark_unarrived(bot_id)
            return True
    return False


# ══════════════════════════════════════════════════════════════════
# STAGE 6: CONVOY FOLLOWER COMMAND
# ══════════════════════════════════════════════════════════════════
def compute_convoy_follow_command(bot_id, follow_target_id, peers_list):
    """Follow the assigned convoy member at convoy_spacing_meters (0.20 m).

    Returns (left, right, trace_dict).

    Rules:
    - If follow target not visible in peers → STOP and wait (never wander).
    - If gap ≤ spacing → STOP (target has stopped or we caught up).
    - Otherwise → approach at speed proportional to gap error.
    - Repulsion from all OTHER peers still applies.
    """
    spacing   = param(bot_id, 'convoy_spacing_meters')
    max_speed = param(bot_id, 'convoy_follow_speed_max')
    gain      = param(bot_id, 'convoy_follow_gain')
    turn_gain = param(bot_id, 'convoy_turn_gain')
    stiction  = param(bot_id, 'motor_stiction_floor')
    max_s     = param(bot_id, 'maximum_motor_speed')

    target_color = bot_record_by_id[follow_target_id]['assigned_color']

    # Locate follow target in this bot's peer detections.
    target_peer = None
    for p in peers_list:
        if p.get('peer_color') == target_color:
            target_peer = p
            break

    if target_peer is None:
        # Target not visible → stop and wait.
        return 0.0, 0.0, {
            'reason': 'target_not_visible',
            'follow_target': follow_target_id,
            'peer_d': None, 'peer_x': None,
            'left': 0.0, 'right': 0.0,
        }

    peer_d = float(target_peer['peer_distance_meters'])
    peer_x = float(target_peer['peer_x_normalized'])

    if peer_d <= spacing:
        # At or inside spacing → stop.
        return 0.0, 0.0, {
            'reason': 'at_spacing',
            'follow_target': follow_target_id,
            'peer_d': peer_d, 'peer_x': peer_x,
            'left': 0.0, 'right': 0.0,
        }

    # Repulsion from all OTHER peers (not from follow target).
    rep_vx, rep_vy, _, _, other_peers = aggregate_peer_repulsion(
        bot_id, peers_list, exclude_color=target_color)
    slowdown, _ = compute_caution_slowdown_factor(bot_id, other_peers)

    # Speed proportional to gap error, clamped.
    error = peer_d - spacing
    speed = max(stiction, min(max_speed, error * gain)) * slowdown
    if 0 < speed < stiction:
        speed = stiction

    # Turn toward follow target.
    turn = max(-1.0, min(1.0, peer_x * turn_gain))

    # Blend repulsion into commands.
    fwd = max(-max_s, min(max_s, speed + rep_vy))
    trn = max(-1.0,   min(1.0,   turn  + rep_vx))

    left, right, _ = arc_wheel_mapping(bot_id, fwd, trn)
    return left, right, {
        'reason': 'following',
        'follow_target': follow_target_id,
        'peer_d': peer_d, 'peer_x': peer_x,
        'slowdown': slowdown,
        'left': left, 'right': right,
    }


# ══════════════════════════════════════════════════════════════════
# STAGE 6: MAIN CONTROLLER
# ══════════════════════════════════════════════════════════════════
def compute_motor_commands(bot_id, cube_detection, peers_list, frame_diff_value=0.0):
    """Stage 6 convoy controller.

    Decision tree
    ─────────────
    1. Stall recovery  (unconditional)
    2. Stall detection
    3. Publish cube distance to brain
    4. Peer critical-stop  (hard safety)
    5. ARRIVED check  (leader only)
    6. Convoy tick → determine role
       A. FOLLOWER → compute_convoy_follow_command
       B. LEADER   → approach target reactively; try handoff when blind
       C. SEEKER   → PSO arc search (pre-announcement only)
    """
    st = bot_state[bot_id]

    # ── 1. Stall recovery ─────────────────────────────────────────
    if st['stall_recovery_frames_left'] > 0:
        st['stall_recovery_frames_left'] -= 1
        st['last_motor_command_left'] = st['last_motor_command_right'] = 0.0
        return 0.0, 0.0, 'stall_recovery', {
            'lock_state': 'STALL_RECOVER', 'role': 'STALL_RECOVER',
            'cube_visible': cube_detection is not None,
            'cube_d': cube_detection['cube_distance_meters'] if cube_detection else None,
            'cube_x': cube_detection['cube_x_normalized']    if cube_detection else None,
            'fill_ratio': compute_frame_fill_ratio(bot_id, cube_detection) if cube_detection else None,
            'peers_summary': [p.get('peer_color','?') for p in peers_list],
            'slowdown_factor': 1.0, 'left': 0.0, 'right': 0.0,
        }

    # ── 2. Stall detection ────────────────────────────────────────
    pixel_th  = param(bot_id, 'stall_detection_pixel_threshold')
    frames_th = param(bot_id, 'stall_detection_frames_threshold')
    last_l, last_r = st['last_motor_command_left'], st['last_motor_command_right']
    commanding = (abs(last_l) > 0.05) or (abs(last_r) > 0.05)
    if commanding and frame_diff_value < pixel_th and cube_detection is None:
        st['stall_low_motion_frames'] += 1
    else:
        st['stall_low_motion_frames'] = 0
    if st['stall_low_motion_frames'] >= frames_th:
        st['stall_low_motion_frames']    = 0
        st['stall_recovery_frames_left'] = param(bot_id, 'stall_recovery_reverse_frames')
        st['last_motor_command_left']    = st['last_motor_command_right'] = 0.0
        print(f"  [{bot_id}] STALL detected (frame_diff={frame_diff_value:.1f})")
        return 0.0, 0.0, 'stall_detected', {
            'lock_state': 'STALL', 'role': 'STALL_RECOVER',
            'cube_visible': False,
            'peers_summary': [p.get('peer_color','?') for p in peers_list],
            'slowdown_factor': 1.0, 'left': 0.0, 'right': 0.0,
        }

    # ── 3. Publish to brain ───────────────────────────────────────
    if cube_detection is not None:
        brain_publish_cube_distance(bot_id, cube_detection['cube_distance_meters'])
        brain_publish_pso_sighting(bot_id,
            cube_detection['cube_x_normalized'],
            cube_detection['cube_distance_meters'])
    else:
        brain_publish_cube_distance(bot_id, None)

    # ── 4. Peer repulsion + critical-stop ─────────────────────────
    peer_rep_vx, peer_rep_vy, critical_peer, peer_summary, relevant_peers = \
        aggregate_peer_repulsion(bot_id, peers_list)
    slowdown_factor, _ = compute_caution_slowdown_factor(bot_id, relevant_peers)

    if critical_peer is not None:
        st['last_motor_command_left'] = st['last_motor_command_right'] = 0.0
        return 0.0, 0.0, 'peer_hard_stop', {
            'lock_state': 'PEER_HOLD', 'role': 'unknown',
            'cube_visible': cube_detection is not None,
            'critical_peer': critical_peer, 'peers_summary': peer_summary,
            'slowdown_factor': slowdown_factor, 'left': 0.0, 'right': 0.0,
        }

    # ── 5. ARRIVED check (only for bot that has reached target) ───
    if st['arrived_by_fill_latched']:
        unarrived = update_unarrival_counters_and_maybe_unarrive(bot_id, cube_detection)
        if not unarrived:
            st['last_motor_command_left'] = st['last_motor_command_right'] = 0.0
            return 0.0, 0.0, 'arrived', {
                'lock_state': 'ARRIVED', 'role': 'ARRIVED',
                'cube_visible': cube_detection is not None,
                'cube_d':   cube_detection['cube_distance_meters'] if cube_detection else None,
                'cube_x':   cube_detection['cube_x_normalized']    if cube_detection else None,
                'fill_ratio': compute_frame_fill_ratio(bot_id, cube_detection) if cube_detection else None,
                'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
                'left': 0.0, 'right': 0.0,
            }

    # ── 6. Convoy tick — determine role ───────────────────────────
    is_leader, announced = brain_convoy_tick(bot_id, cube_detection is not None)
    _, convoy_leader_id, convoy_chain = brain_get_convoy_info()
    is_leader = (convoy_leader_id == bot_id) and announced  # re-check after possible announcement

    # ╔══════════════════════════════════════════════════════════════╗
    # ║ 6A. FOLLOWER — convoy announced, this bot is NOT the leader ║
    # ╚══════════════════════════════════════════════════════════════╝
    if announced and not is_leader:
        my_idx = convoy_chain.index(bot_id) if bot_id in convoy_chain else -1
        # Follow the bot one slot ahead; fall back to leader if chain lookup fails.
        follow_target_id = convoy_chain[my_idx - 1] if my_idx > 0 else convoy_leader_id

        left, right, ftrace = compute_convoy_follow_command(
            bot_id, follow_target_id, peers_list)
        follow_reason = ftrace.get('reason', '?')

        # Track loss of the bot ahead so we can SEARCH instead of freezing
        # (restores continuous search when a convoy member goes out of view).
        if follow_reason == 'target_not_visible':
            st['follower_lost_target_frames'] += 1
        else:
            st['follower_lost_target_frames'] = 0

        search_grace = param(bot_id, 'convoy_follower_search_grace_frames')
        if (follow_reason == 'target_not_visible'
                and st['follower_lost_target_frames'] > search_grace):
            # Lost the member ahead for too long -> search to re-acquire it / the
            # target. If we find the cube while the leader is blind, the brain
            # handoff promotes us automatically.
            if cube_detection is not None:
                st['last_seen_cube_x_normalized'] = cube_detection['cube_x_normalized']
            current_x = st['last_seen_cube_x_normalized']
            pso_vx, _ = brain_compute_pso_velocity(bot_id, current_x, 9999.0)
            fwd   = param(bot_id, 'search_forward_speed')
            rot   = param(bot_id, 'lost_rotation_speed')
            max_s = param(bot_id, 'maximum_motor_speed')
            turn_dir = (math.copysign(1.0, pso_vx) if abs(pso_vx) > 0.02
                        else (1.0 if current_x >= 0 else -1.0))
            left  = min(max_s, fwd + rot * turn_dir)
            right = max(0.0,   fwd - rot * turn_dir)
            rotate_dir = 'right' if turn_dir > 0 else 'left'
            st['last_motor_command_left'], st['last_motor_command_right'] = left, right
            return left, right, f'follower_searching_{rotate_dir}', {
                'lock_state': 'SEARCHING', 'role': 'FOLLOWER',
                'cube_visible': cube_detection is not None,
                'cube_d': cube_detection['cube_distance_meters'] if cube_detection else None,
                'cube_x': cube_detection['cube_x_normalized']    if cube_detection else None,
                'fill_ratio': compute_frame_fill_ratio(bot_id, cube_detection) if cube_detection else None,
                'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
                'follow_target': follow_target_id, 'follow_reason': 'searching',
                'lost_frames': st['follower_lost_target_frames'],
                'rotate_dir': rotate_dir, 'convoy_chain': convoy_chain,
                'left': left, 'right': right,
            }

        st['last_motor_command_left']  = left
        st['last_motor_command_right'] = right
        if cube_detection is not None:
            st['last_seen_cube_x_normalized'] = cube_detection['cube_x_normalized']
            st['consecutive_missed_frames']   = 0

        return left, right, f'convoy_follow_{follow_reason}', {
            'lock_state':     'CONVOY_FOLLOW',
            'role':           'FOLLOWER',
            'cube_visible':   cube_detection is not None,
            'cube_d':   cube_detection['cube_distance_meters'] if cube_detection else None,
            'cube_x':   cube_detection['cube_x_normalized']    if cube_detection else None,
            'fill_ratio': compute_frame_fill_ratio(bot_id, cube_detection) if cube_detection else None,
            'peers_summary':  peer_summary,
            'slowdown_factor': ftrace.get('slowdown', 1.0),
            'follow_target':  follow_target_id,
            'follow_reason':  follow_reason,
            'follow_peer_d':  ftrace.get('peer_d'),
            'convoy_chain':   convoy_chain,
            'left': left, 'right': right,
        }

    # ╔══════════════════════════════════════════════════════════════╗
    # ║ 6B. LEADER — approach target reactively                     ║
    # ╚══════════════════════════════════════════════════════════════╝
    if announced and is_leader:

        if cube_detection is not None:
            # ── Leader sees target ─────────────────────────────────
            st['consecutive_missed_frames'] = 0
            d = cube_detection['cube_distance_meters']   # RAW for immediate reaction (req 2)
            x = cube_detection['cube_x_normalized']
            st['last_seen_cube_x_normalized'] = x
            st['previous_cube_distance']      = d
            fill_ratio = compute_frame_fill_ratio(bot_id, cube_detection)

            # Arrival check — require BOTH a large bbox AND a close distance,
            # so a far target (or noisy/large bbox) can never (re-)latch ARRIVED.
            arrive_max_d = param(bot_id, 'unarrive_distance_threshold_meters')
            if (fill_ratio is not None
                    and fill_ratio > param(bot_id, 'arrived_cube_fills_frame_ratio')
                    and d <= arrive_max_d):
                blocking = any(
                    float(p.get('peer_distance_meters', 999)) < param(bot_id,'convoy_spacing_meters') + 0.10
                    and abs(float(p.get('peer_x_normalized', 0))) < 0.35
                    for p in peers_list)
                if not blocking:
                    st['arrived_by_fill_latched']    = True
                    st['frames_cube_departed_count'] = 0
                    st['frames_cube_missing_count']  = 0
                    brain_mark_arrived(bot_id)
                    print(f"  [{bot_id}] ARRIVED (fill={fill_ratio:.2f})")
                    st['last_motor_command_left'] = st['last_motor_command_right'] = 0.0
                    return 0.0, 0.0, 'arrived_by_fill', {
                        'lock_state': 'ARRIVED', 'role': 'ARRIVED',
                        'cube_visible': True, 'cube_d': d, 'cube_x': x,
                        'fill_ratio': fill_ratio,
                        'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
                        'convoy_chain': convoy_chain, 'left': 0.0, 'right': 0.0,
                    }

            # Approach using RAW distance — reacts immediately when target moves.
            stop_dist  = param(bot_id, 'target_stopping_distance_meters')
            zone       = param(bot_id, 'approach_complete_zone_meters')
            bear_tol   = param(bot_id, 'approach_bearing_tolerance')
            bear_tol_f = param(bot_id, 'approach_bearing_tolerance_final')
            close_fill = param(bot_id, 'close_range_fill_ratio')
            close_bear = param(bot_id, 'close_range_bearing_tolerance')
            creep      = param(bot_id, 'forward_creep_when_bearing_off')
            max_speed  = param(bot_id, 'maximum_motor_speed')
            stiction   = param(bot_id, 'motor_stiction_floor')
            min_move   = param(bot_id, 'forward_minimum_to_move')

            dist_error   = d - stop_dist
            dist_in_zone = abs(dist_error) < zone
            active_bear_tol = (close_bear if (fill_ratio is not None and fill_ratio > close_fill)
                               else (bear_tol_f if dist_in_zone else bear_tol))
            bearing_ok = abs(x) < active_bear_tol

            if dist_in_zone and bearing_ok:
                forward_target, sub_state = 0.0, 'arrived'
            elif dist_in_zone:
                forward_target, sub_state = creep, 'centering'
            else:
                fp = max(-1.0, min(1.0, param(bot_id,'forward_pull_gain_per_meter') * dist_error))
                if abs(fp) < min_move:
                    forward_target = 0.0
                elif fp > 0:
                    forward_target = min(max_speed, stiction + fp*(max_speed - stiction))
                else:
                    forward_target = max(-max_speed, -stiction + fp*(max_speed - stiction))
                sub_state = 'tracking'

            if forward_target > 0:
                forward_target = max(stiction, forward_target * slowdown_factor)

            tp = param(bot_id,'turn_pull_gain_per_x_normalized') * x
            cube_turn = (0.0 if abs(tp) < param(bot_id,'turn_deadband_in_local_x')
                         else max(-1.0, min(1.0, tp * param(bot_id,'turn_to_wheel_diff_scale'))))

            fwd_cmd = max(-max_speed, min(max_speed, forward_target + peer_rep_vy))
            trn_cmd = max(-1.0,       min(1.0,       cube_turn      + peer_rep_vx))

            left, right, arc_ratio = arc_wheel_mapping(bot_id, fwd_cmd, trn_cmd)
            st['last_motor_command_left']  = left
            st['last_motor_command_right'] = right
            return left, right, sub_state, {
                'lock_state':    'LOCKED',
                'role':          'CONVOY_LEADER',
                'cube_visible':  True,
                'cube_d': d, 'cube_x': x, 'fill_ratio': fill_ratio,
                'forward_target': fwd_cmd, 'turn_intensity': trn_cmd,
                'arc_ratio': arc_ratio, 'slowdown_factor': slowdown_factor,
                'peers_summary': peer_summary, 'convoy_chain': convoy_chain,
                'active_bear_tol': active_bear_tol,
                'left': left, 'right': right,
            }

        else:
            # ── Leader lost target ─────────────────────────────────
            st['consecutive_missed_frames'] += 1
            missed = st['consecutive_missed_frames']

            # Try leadership handoff if blind long enough (req 3).
            new_leader = brain_convoy_try_handoff(bot_id, missed)
            if new_leader is not None:
                # Handed off — this bot becomes a follower next frame.
                st['last_motor_command_left'] = st['last_motor_command_right'] = 0.0
                return 0.0, 0.0, 'leader_handoff', {
                    'lock_state': 'HANDOFF', 'role': 'LEADER_HANDOFF',
                    'cube_visible': False, 'peers_summary': peer_summary,
                    'slowdown_factor': slowdown_factor,
                    'new_leader': new_leader, 'convoy_chain': convoy_chain,
                    'left': 0.0, 'right': 0.0,
                }

            # Grace frames: continue last motor command.
            grace_max = param(bot_id, 'cube_lock_grace_frames')
            if missed <= grace_max:
                ll, lr = st['last_motor_command_left'], st['last_motor_command_right']
                approx_fwd  = (ll + lr) / 2.0
                approx_turn = (ll - lr) / 2.0
                if approx_fwd > 0:
                    approx_fwd *= slowdown_factor
                fwd_g = max(-param(bot_id,'maximum_motor_speed'),
                            min(param(bot_id,'maximum_motor_speed'), approx_fwd + peer_rep_vy))
                trn_g = max(-1.0, min(1.0, approx_turn + peer_rep_vx))
                left, right, _ = arc_wheel_mapping(bot_id, fwd_g, trn_g)
                st['last_motor_command_left'], st['last_motor_command_right'] = left, right
                return left, right, 'lock_grace', {
                    'lock_state': 'GRACE', 'role': 'CONVOY_LEADER',
                    'cube_visible': False, 'grace_frames_used': missed,
                    'last_seen_x': st['last_seen_cube_x_normalized'],
                    'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
                    'convoy_chain': convoy_chain, 'left': left, 'right': right,
                }

            # Past grace — PSO arc search while still leader.
            current_x = st['last_seen_cube_x_normalized']
            pso_vx, _ = brain_compute_pso_velocity(bot_id, current_x, 9999.0)
            fwd   = param(bot_id, 'search_forward_speed')
            rot   = param(bot_id, 'lost_rotation_speed')
            max_s = param(bot_id, 'maximum_motor_speed')
            turn_dir = (math.copysign(1.0, pso_vx) if abs(pso_vx) > 0.02
                        else (1.0 if current_x >= 0 else -1.0))
            left  = min(max_s, fwd + rot * turn_dir)
            right = max(0.0,   fwd - rot * turn_dir)
            rotate_dir = 'right' if turn_dir > 0 else 'left'
            st['last_motor_command_left'], st['last_motor_command_right'] = left, right
            return left, right, f'leader_searching_{rotate_dir}', {
                'lock_state': 'LOST', 'role': 'CONVOY_LEADER',
                'cube_visible': False, 'grace_frames_used': missed,
                'last_seen_x': current_x, 'rotate_dir': rotate_dir,
                'pso_vx': round(pso_vx, 3),
                'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
                'convoy_chain': convoy_chain, 'left': left, 'right': right,
            }

    # ╔══════════════════════════════════════════════════════════════╗
    # ║ 6C. SEEKER — convoy not yet announced; PSO arc search        ║
    # ╚══════════════════════════════════════════════════════════════╝
    if cube_detection is not None:
        # Bot sees cube but announce threshold not yet reached.
        st['consecutive_missed_frames'] = 0
        x    = cube_detection['cube_x_normalized']
        d    = cube_detection['cube_distance_meters']
        st['last_seen_cube_x_normalized'] = x
        fill = compute_frame_fill_ratio(bot_id, cube_detection)
        tl, tr = compute_tracker_rotation(bot_id, x, fill)
        st['last_motor_command_left'], st['last_motor_command_right'] = tl, tr
        return tl, tr, 'seeker_tracking', {
            'lock_state': 'SEEKING', 'role': 'SEEKER',
            'cube_visible': True, 'cube_d': d, 'cube_x': x, 'fill_ratio': fill,
            'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
            'left': tl, 'right': tr,
        }

    # Cube not visible — PSO arc.
    st['consecutive_missed_frames'] += 1
    current_x = st['last_seen_cube_x_normalized']
    pso_vx, _ = brain_compute_pso_velocity(bot_id, current_x, 9999.0)
    fwd   = param(bot_id, 'search_forward_speed')
    rot   = param(bot_id, 'lost_rotation_speed')
    max_s = param(bot_id, 'maximum_motor_speed')
    turn_dir  = (math.copysign(1.0, pso_vx) if abs(pso_vx) > 0.02
                 else (1.0 if current_x >= 0 else -1.0))
    left  = min(max_s, fwd + rot * turn_dir)
    right = max(0.0,   fwd - rot * turn_dir)
    rotate_dir = 'right' if turn_dir > 0 else 'left'
    st['last_motor_command_left'], st['last_motor_command_right'] = left, right
    return left, right, f'searching_{rotate_dir}', {
        'lock_state': 'SEARCHING', 'role': 'SEEKER',
        'cube_visible': False, 'last_seen_x': current_x,
        'rotate_dir': rotate_dir, 'pso_vx': round(pso_vx, 3),
        'peers_summary': peer_summary, 'slowdown_factor': slowdown_factor,
        'left': left, 'right': right,
    }


print('Controller defined (Stage 6: convoy system).')

## §6. Tracking loops

One thread per bot. Each thread runs an independent control loop. Threads don't talk to each other.


In [ ]:
tracking_should_run_by_bot_id = {bid: False for bid in known_bot_ids}
loop_status_lock              = threading.Lock()
loop_status_by_bot_id         = {bid: {} for bid in known_bot_ids}
iteration_count_by_bot_id     = {bid: 0  for bid in known_bot_ids}


def run_tracking_loop(bot_id):
    iteration_count_by_bot_id[bot_id] = 0
    period    = param(bot_id, 'control_loop_period_seconds')
    log_every = param(bot_id, 'log_every_n_iterations')
    print(f"[{bot_id}] tracking start")
    while tracking_should_run_by_bot_id[bot_id]:
        iter_start = time.time()
        ok, cube, peers, latency, frame_diff = fetch_state(bot_id)
        if not ok:
            time.sleep(period)
            continue
        left, right, state_label, trace = compute_motor_commands(bot_id, cube, peers, frame_diff)
        send_command(bot_id, left, right)
        iteration_count_by_bot_id[bot_id] += 1
        n = iteration_count_by_bot_id[bot_id]
        with loop_status_lock:
            loop_status_by_bot_id[bot_id] = {
                **trace, 'state': state_label,
                'network_latency_ms': latency * 1000.0,
                'iteration': n,
            }
        if log_every and n % log_every == 0:
            peers_s  = ','.join(trace.get('peers_summary', [])) or '-'
            role     = trace.get('role', '?')
            cube_str = (f"d={trace.get('cube_d', 0):.2f}"
                        if trace.get('cube_visible') else 'cube=NO')
            fill     = trace.get('fill_ratio')
            fill_str = f" fill={fill:.2f}" if fill is not None else ''
            chain    = trace.get('convoy_chain', [])
            chain_s  = '→'.join(b.replace('bot_','') for b in chain) if chain else '-'
            ft       = trace.get('follow_target', '')
            ft_s     = f" ->{ft.replace('bot_','')}" if ft else ''
            print(f"  [{bot_id}] i={n:4d} [{role}]{ft_s} {state_label:<28s} "
                  f"{cube_str}{fill_str} chain=[{chain_s}] peers=[{peers_s}] "
                  f"→ L={left:+.2f} R={right:+.2f}")
        elapsed = time.time() - iter_start
        if elapsed < period:
            time.sleep(period - elapsed)
    stop_bot(bot_id)
    print(f"[{bot_id}] tracking stopped. iterations={iteration_count_by_bot_id[bot_id]}")


def start_tracking_for_bot(bot_id):
    if tracking_should_run_by_bot_id[bot_id]:
        print(f"[{bot_id}] already running.")
        return
    tracking_should_run_by_bot_id[bot_id] = True
    threading.Thread(target=run_tracking_loop, args=(bot_id,), daemon=True).start()


def start_tracking_for_all_bots():
    reset_bot_states_for_new_run()
    brain_reset()
    for bid in known_bot_ids:
        start_tracking_for_bot(bid)


def stop_tracking_for_all_bots():
    for bid in known_bot_ids:
        tracking_should_run_by_bot_id[bid] = False
    time.sleep(0.3)
    stop_all_bots()
    print('All bots stopped.')


print('Tracking loops ready (Stage 6).')

## §7. Multi-pane debug view

One pane per bot. Shows `/camera_annotated` (cube + peer boxes drawn by the bot) and a per-bot status line.

**Use this BEFORE starting tracking** to confirm each bot detects what it should:
- Each bot's pane should show a red box around the cube (when it's in that bot's view).
- Each bot's pane should show colored boxes around other bots' cylinders (when peers are in that bot's view).
- The status line shows peer detections with distances.


In [ ]:
TRANSPARENT_PNG = base64.b64decode(
    'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNkAAIAAAoAAv/lxKUAAAAASUVORK5CYII=')

pane_width  = 400
pane_height = 220

debug_image_widget_by_bot_id = {}
debug_status_label_by_bot_id = {}

for b in bot_registry:
    bid = b['bot_id']
    debug_image_widget_by_bot_id[bid] = widgets.Image(
        value=TRANSPARENT_PNG, format='png', width=pane_width, height=pane_height)
    debug_status_label_by_bot_id[bid] = widgets.Label(value='(idle)')

def make_pane(b):
    bid   = b['bot_id']
    hex_c = display_color_hex_for_color.get(b['assigned_color'], '#000')
    title = widgets.HTML(
        f"<b>{bid}</b>  <span style='color:{hex_c};font-weight:bold;'>● {b['assigned_color']}</span>  "
        f"<span style='color:#777;font-size:0.85em;'>{b['ip_address']}</span>")
    return widgets.VBox([title, debug_image_widget_by_bot_id[bid],
                          debug_status_label_by_bot_id[bid]])

panes = [make_pane(b) for b in bot_registry]
if len(panes) == 1:
    grid = panes[0]
elif len(panes) == 2:
    grid = widgets.HBox(panes)
else:
    grid = widgets.VBox([widgets.HBox(panes[:2]),
                          widgets.HBox(panes[2:4]) if len(panes) >= 3 else widgets.HBox([])])
display(grid)

debug_view_should_run = True
debug_camera_period   = 0.25
debug_label_period    = 0.15

def _load_overlay_font():
    font_paths_to_try = [
        '/System/Library/Fonts/Supplemental/Arial Bold.ttf',
        '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
        '/usr/share/fonts/dejavu/DejaVuSans-Bold.ttf',
        'C:\\\\Windows\\\\Fonts\\\\arialbd.ttf',
    ]
    from PIL import ImageFont
    for p in font_paths_to_try:
        try:
            return ImageFont.truetype(p, size=20)
        except (IOError, OSError):
            continue
    return ImageFont.load_default()

OVERLAY_FONT = _load_overlay_font()

# Stage 6: updated role palette
ROLE_COLORS = {
    'CONVOY_LEADER':  ( 60, 240,  60),   # bright green  — leader chasing target
    'FOLLOWER':       (100, 180, 255),   # light blue    — following chain member
    'SEEKER':         (255, 190,  50),   # amber         — searching pre-announcement
    'ARRIVED':        ( 80, 160, 255),   # blue          — reached target
    'LEADER_HANDOFF': (255, 230,  50),   # yellow        — handing off leadership
    'STALL_RECOVER':  (255,  60,  60),   # red           — stall recovery
    'unknown':        (200, 200, 200),
}

def overlay_distance_text_on_jpeg(jpeg_bytes, bot_status_snapshot):
    if not jpeg_bytes:
        return jpeg_bytes
    try:
        img = PILImage.open(io.BytesIO(jpeg_bytes)).convert('RGB')
    except Exception:
        return jpeg_bytes
    draw   = ImageDraw.Draw(img)
    lines  = []
    colors = []

    role       = bot_status_snapshot.get('role', '?')
    role_color = ROLE_COLORS.get(role, (255, 255, 255))
    lines.append(f"-- {role} --")
    colors.append(role_color)

    # Convoy chain (very useful to see at a glance).
    chain = bot_status_snapshot.get('convoy_chain', [])
    if chain:
        chain_s = ' > '.join(b.replace('bot_', '') for b in chain)
        lines.append(f"chain  {chain_s}")
        colors.append((160, 255, 160))

    # Follow target (FOLLOWER role).
    ft = bot_status_snapshot.get('follow_target')
    if ft:
        pd   = bot_status_snapshot.get('follow_peer_d')
        pd_s = f"{pd:.2f}m" if pd is not None else '?'
        lines.append(f"-> {ft.replace('bot_','')}  {pd_s}")
        colors.append((100, 200, 255))

    if bot_status_snapshot.get('cube_visible'):
        d = bot_status_snapshot.get('cube_d')
        if d is not None:
            lines.append(f"CUBE   {d:.2f} m")
            colors.append((255, 80, 80))
        x = bot_status_snapshot.get('cube_x')
        if x is not None:
            lines.append(f"bear   {x:+.2f}")
            colors.append((255, 200, 200))
        fill = bot_status_snapshot.get('fill_ratio')
        if fill is not None:
            lines.append(f"fill   {fill:.2f}")
            colors.append((255, 220, 60))
    else:
        lines.append('cube   NOT VISIBLE')
        colors.append((180, 180, 180))

    peer_rgb_for_color = {
        'red':    (255,  80,  80), 'orange': (255, 140,  40),
        'yellow': (255, 220,  40), 'blue':   ( 80, 160, 255),
        'purple': (200, 100, 220),
    }
    for peer_str in bot_status_snapshot.get('peers_summary', []):
        try:
            color_part, dist_part = peer_str.split('@')
            lines.append(f"{color_part[:3]:<4s}  {dist_part}")
            colors.append(peer_rgb_for_color.get(color_part, (255, 255, 255)))
        except (ValueError, AttributeError):
            continue

    slow = bot_status_snapshot.get('slowdown_factor', 1.0)
    if slow < 0.95:
        lines.append(f"slow   {slow:.2f}")
        colors.append((255, 160, 80))

    lock = bot_status_snapshot.get('lock_state')
    if lock in ('PEER_HOLD', 'STALL'):
        lines.append(f'[{lock}]')
        colors.append((255, 60, 60))

    pad         = 6
    line_height = 24
    box_w       = 0
    for line in lines:
        try:
            tb    = draw.textbbox((0, 0), line, font=OVERLAY_FONT)
            box_w = max(box_w, tb[2] - tb[0])
        except AttributeError:
            box_w = max(box_w, 8 * len(line))
    box_h = line_height * len(lines) + pad
    draw.rectangle([0, 0, box_w + 2 * pad, box_h], fill=(0, 0, 0))
    for idx, (line, rgb) in enumerate(zip(lines, colors)):
        draw.text((pad, pad // 2 + idx * line_height), line, fill=rgb, font=OVERLAY_FONT)

    out = io.BytesIO()
    img.save(out, format='JPEG', quality=80)
    return out.getvalue()


def run_debug_camera_loop_for_bot(bot_id):
    record  = bot_record_by_id[bot_id]
    session = http_session_by_bot_id[bot_id]
    while debug_view_should_run:
        try:
            r = session.get(urljoin(base_url_for_bot(record), '/camera_annotated'), timeout=1.5)
            if r.status_code == 200 and r.content:
                with loop_status_lock:
                    status_snap = dict(loop_status_by_bot_id.get(bot_id, {}))
                overlaid_bytes = overlay_distance_text_on_jpeg(r.content, status_snap)
                debug_image_widget_by_bot_id[bot_id].format = 'jpeg'
                debug_image_widget_by_bot_id[bot_id].value  = overlaid_bytes
        except requests.RequestException:
            pass
        time.sleep(debug_camera_period)


def run_debug_label_loop():
    while debug_view_should_run:
        with loop_status_lock:
            snap = dict(loop_status_by_bot_id)
        brain_snap = brain_get_status_snapshot()
        chain_str  = '→'.join(b.replace('bot_','') for b in brain_snap.get('chain', []))
        announced  = brain_snap.get('announced', False)
        for bid, st in snap.items():
            if not st:
                debug_status_label_by_bot_id[bid].value = '(idle)'
                continue
            role    = st.get('role', '?')
            state   = st.get('state', '?')
            cube_s  = (f"d={st.get('cube_d', 0):.2f}m" if st.get('cube_visible') else 'cube=NO')
            fill    = st.get('fill_ratio')
            fill_s  = f" fill={fill:.2f}" if fill is not None else ''
            peers_s = ','.join(st.get('peers_summary', [])) or '-'
            L, R    = st.get('left', 0), st.get('right', 0)
            ft      = st.get('follow_target', '')
            ft_s    = f" ->{ft.replace('bot_','')}" if ft else ''
            ann_s   = 'CONVOY' if announced else 'SEARCH'
            debug_status_label_by_bot_id[bid].value = (
                f"[{role}]{ft_s} {state} | {cube_s}{fill_s} | "
                f"{ann_s}:[{chain_str}] peers:{peers_s} | L={L:+.2f} R={R:+.2f}")
        time.sleep(debug_label_period)


for bid in known_bot_ids:
    threading.Thread(target=run_debug_camera_loop_for_bot, args=(bid,), daemon=True).start()
threading.Thread(target=run_debug_label_loop, daemon=True).start()
print('Debug view started (Stage 6 — convoy role colours + chain display).')

### §7.1 Stop debug view

In [ ]:
debug_view_should_run = False
print('Debug view stop requested.')


## §8. Start / Stop tracking

**Before §8.1:**
- Place the red cube in a sensible spot.
- Place the bots so they can ROUGHLY see the cube. (They'll find it if they don't.)
- Look at the debug view in §7 — confirm each bot sees the cube (red box).


### §8.1 START tracking

In [ ]:
start_tracking_for_all_bots()


### §8.2 STOP tracking

In [ ]:
stop_tracking_for_all_bots()


## §9. Shutdown

In [ ]:
stop_tracking_for_all_bots()
debug_view_should_run = False
time.sleep(0.3)
for s in http_session_by_bot_id.values():
    s.close()
print('Shutdown complete.')
